In [1]:
import numpy as np
import joblib
import requests
import warnings

from datetime import datetime

warnings.filterwarnings("ignore")


# ============================================================
# BLYNK CONFIGURATION
# ============================================================

BLYNK_TOKEN = "K9HQ61rMsnDEf9OESEUX04EOlU3LtKQL"

BASE_URL = "https://blynk.cloud/external/api"


# ============================================================
# BLYNK VIRTUAL PINS
# ============================================================

V0_URL = f"{BASE_URL}/get?token={BLYNK_TOKEN}&V0"
V1_URL = f"{BASE_URL}/get?token={BLYNK_TOKEN}&V1"
V2_URL = f"{BASE_URL}/get?token={BLYNK_TOKEN}&V2"
V3_URL = f"{BASE_URL}/get?token={BLYNK_TOKEN}&V3"
V4_URL = f"{BASE_URL}/get?token={BLYNK_TOKEN}&V4"
V5_URL = f"{BASE_URL}/get?token={BLYNK_TOKEN}&V5"


# ============================================================
# PRODUCT CONFIGURATION
# ============================================================
#
# These dates are default/static product information.
#
# You can change these dates according to your products.
#
# Format:
# YYYY-MM-DD
#
# ============================================================

PRODUCT_CONFIG = {

    "PRODUCT 1": {

        "manufacturing_date": "2025-01-15",

        "installation_date": "2025-02-01"

    },

    "PRODUCT 2": {

        "manufacturing_date": "2025-03-10",

        "installation_date": "2025-04-05"

    },

    "PRODUCT 3": {

        "manufacturing_date": "2025-06-20",

        "installation_date": "2025-07-01"

    }

}


# ============================================================
# MODEL FILES
# ============================================================

SCALER_FILE = "scaler.pkl"

LABEL_ENCODER_FILE = "label_encoder.pkl"

CONDITION_MODEL_FILE = "condition_c.pkl"

LIFE_MODEL_FILE = "life_r.pkl"


# ============================================================
# LOAD ML MODELS
# ============================================================

print("\n" + "=" * 70)
print("LOADING COMMON ML MODELS")
print("=" * 70)


try:

    scaler = joblib.load(
        SCALER_FILE
    )

    label_encoder = joblib.load(
        LABEL_ENCODER_FILE
    )

    condition_model = joblib.load(
        CONDITION_MODEL_FILE
    )

    life_model = joblib.load(
        LIFE_MODEL_FILE
    )


    print("✅ Scaler loaded successfully")

    print("✅ Label Encoder loaded successfully")

    print("✅ Condition Model loaded successfully")

    print("✅ RUL / Life Model loaded successfully")


except FileNotFoundError as e:

    print("\n❌ MODEL FILE NOT FOUND")

    print(f"Missing file: {e.filename}")

    print("\nRequired files:")

    print("1. scaler.pkl")

    print("2. label_encoder.pkl")

    print("3. condition.pkl")

    print("4. life.pkl")

    print("\nMake sure these files are inside the same folder.")

    exit()


except Exception as e:

    print("\n❌ MODEL LOADING ERROR")

    print(str(e))

    exit()


# ============================================================
# FETCH BLYNK SENSOR VALUE
# ============================================================

def fetch_sensor(url, sensor_name):

    try:

        response = requests.get(
            url,
            timeout=5
        )


        # ----------------------------------------------------
        # HTTP ERROR
        # ----------------------------------------------------

        if response.status_code != 200:

            print(
                f"❌ {sensor_name}: "
                f"HTTP {response.status_code}"
            )

            print(
                f"Response: {response.text}"
            )

            return None


        # ----------------------------------------------------
        # RAW VALUE
        # ----------------------------------------------------

        raw_value = response.text.strip()


        if raw_value == "":

            print(
                f"❌ {sensor_name}: Empty Blynk value"
            )

            return None


        # ----------------------------------------------------
        # CONVERT TO FLOAT
        # ----------------------------------------------------

        value = float(
            raw_value
        )


        print(
            f"✅ {sensor_name:<25} : {value}"
        )


        return value


    except requests.exceptions.Timeout:

        print(
            f"❌ {sensor_name}: "
            f"Blynk request timeout"
        )

        return None


    except requests.exceptions.ConnectionError:

        print(
            f"❌ {sensor_name}: "
            f"Internet / connection error"
        )

        return None


    except ValueError:

        print(
            f"❌ {sensor_name}: "
            f"Non-numeric Blynk value"
        )

        return None


    except Exception as e:

        print(
            f"❌ {sensor_name}: {e}"
        )

        return None


# ============================================================
# FETCH COMMON SENSOR VALUES
# ============================================================

print("\n" + "=" * 70)

print("FETCHING COMMON SENSOR VALUES")

print("=" * 70)


# ------------------------------------------------------------
# V0 = VOLTAGE
# ------------------------------------------------------------

voltage = fetch_sensor(
    V0_URL,
    "Common Voltage (V0)"
)


# ------------------------------------------------------------
# V4 = TEMPERATURE
# ------------------------------------------------------------

temperature = fetch_sensor(
    V4_URL,
    "Common Temperature (V4)"
)


# ------------------------------------------------------------
# V5 = VIBRATION
# ------------------------------------------------------------

vibration = fetch_sensor(
    V5_URL,
    "Common Vibration (V5)"
)


# ============================================================
# FETCH PRODUCT CURRENT VALUES
# ============================================================

print("\n" + "=" * 70)

print("FETCHING PRODUCT CURRENT VALUES")

print("=" * 70)


# ------------------------------------------------------------
# V1 = PRODUCT 1 CURRENT
# ------------------------------------------------------------

product1_current = fetch_sensor(
    V1_URL,
    "Product 1 Current (V1)"
)


# ------------------------------------------------------------
# V2 = PRODUCT 2 CURRENT
# ------------------------------------------------------------

product2_current = fetch_sensor(
    V2_URL,
    "Product 2 Current (V2)"
)


# ------------------------------------------------------------
# V3 = PRODUCT 3 CURRENT
# ------------------------------------------------------------

product3_current = fetch_sensor(
    V3_URL,
    "Product 3 Current (V3)"
)


# ============================================================
# VALIDATE COMMON VALUES
# ============================================================

common_values = [

    voltage,

    temperature,

    vibration

]


if any(
    value is None
    for value in common_values
):

    print("\n" + "=" * 70)

    print("❌ COMMON SENSOR FETCH FAILED")

    print("=" * 70)

    print("""

Please check the following:

1. Blynk device is ONLINE
2. Blynk token is correct
3. V0 datastream exists
4. V4 datastream exists
5. V5 datastream exists
6. Datastream values are numeric
7. Internet connection is available

Current values:

V0 Voltage       = {}
V4 Temperature   = {}
V5 Vibration     = {}

""".format(
        voltage,
        temperature,
        vibration
    ))

    exit()


# ============================================================
# VALIDATE PRODUCT CURRENT VALUES
# ============================================================

product_currents = [

    product1_current,

    product2_current,

    product3_current

]


if any(
    value is None
    for value in product_currents
):

    print("\n" + "=" * 70)

    print("❌ PRODUCT CURRENT FETCH FAILED")

    print("=" * 70)

    print("""

Please check:

Product 1 → V1
Product 2 → V2
Product 3 → V3

Make sure all three Blynk datastreams exist.

Current values:

Product 1 Current = {}
Product 2 Current = {}
Product 3 Current = {}

""".format(
        product1_current,
        product2_current,
        product3_current
    ))

    exit()


# ============================================================
# DISPLAY COMMON VALUES
# ============================================================

print("\n" + "=" * 70)

print("COMMON SENSOR VALUES")

print("=" * 70)


print(
    f"Voltage       : "
    f"{voltage:.2f} V"
)


print(
    f"Temperature   : "
    f"{temperature:.2f} °C"
)


print(
    f"Vibration     : "
    f"{vibration:.4f}"
)


# ============================================================
# DISPLAY PRODUCT CURRENT VALUES
# ============================================================

print("\n" + "=" * 70)

print("PRODUCT LOAD CURRENT VALUES")

print("=" * 70)


print(
    f"Product 1 Current : "
    f"{product1_current:.2f}"
)


print(
    f"Product 2 Current : "
    f"{product2_current:.2f}"
)


print(
    f"Product 3 Current : "
    f"{product3_current:.2f}"
)


# ============================================================
# CALCULATE PRODUCT AGE
# ============================================================

def calculate_product_age(
    manufacturing_date,
    installation_date
):

    try:

        today = datetime.today()


        manufacture_dt = datetime.strptime(
            manufacturing_date,
            "%Y-%m-%d"
        )


        installation_dt = datetime.strptime(
            installation_date,
            "%Y-%m-%d"
        )


        # ----------------------------------------------------
        # VALIDATION
        # ----------------------------------------------------

        if installation_dt < manufacture_dt:

            raise ValueError(
                "Installation date cannot be "
                "before manufacturing date."
            )


        # ----------------------------------------------------
        # DAYS
        # ----------------------------------------------------

        manufacturing_age_days = (
            today - manufacture_dt
        ).days


        installation_age_days = (
            today - installation_dt
        ).days


        # ----------------------------------------------------
        # YEARS
        # ----------------------------------------------------

        manufacturing_age_years = (

            manufacturing_age_days
            / 365.25

        )


        installation_age_years = (

            installation_age_days
            / 365.25

        )


        return {

            "manufacturing_age_days":
                manufacturing_age_days,

            "installation_age_days":
                installation_age_days,

            "manufacturing_age_years":
                manufacturing_age_years,

            "installation_age_years":
                installation_age_years

        }


    except Exception as e:

        print(
            f"❌ Date calculation error: {e}"
        )

        return {

            "manufacturing_age_days": 0,

            "installation_age_days": 0,

            "manufacturing_age_years": 0,

            "installation_age_years": 0

        }


# ============================================================
# PREDICTION FUNCTION
# ============================================================

def predict_product(
    product_name,
    load_current
):


    print("\n")

    print("-" * 70)

    print(
        f"🔮 PREDICTING {product_name}"
    )

    print("-" * 70)


    # ========================================================
    # PRODUCT DATE INFORMATION
    # ========================================================

    product_info = PRODUCT_CONFIG[
        product_name
    ]


    manufacturing_date = (

        product_info[
            "manufacturing_date"
        ]

    )


    installation_date = (

        product_info[
            "installation_date"
        ]

    )


    # ========================================================
    # CALCULATE AGE
    # ========================================================

    age_info = calculate_product_age(

        manufacturing_date,

        installation_date

    )


    manufacturing_age_years = (

        age_info[
            "manufacturing_age_years"
        ]

    )


    installation_age_years = (

        age_info[
            "installation_age_years"
        ]

    )


    # ========================================================
    # MODEL INPUT
    # ========================================================

    input_data = np.array([

        [

            voltage,

            load_current,

            temperature,

            vibration

        ]

    ])


    print("\nModel Input")

    print("-" * 50)


    print(
        f"Voltage       : "
        f"{voltage:.2f}"
    )


    print(
        f"Load Current  : "
        f"{load_current:.2f}"
    )


    print(
        f"Temperature   : "
        f"{temperature:.2f}"
    )


    print(
        f"Vibration     : "
        f"{vibration:.4f}"
    )


    # ========================================================
    # SCALE DATA
    # ========================================================

    try:

        scaled_data = scaler.transform(
            input_data
        )

    except Exception as e:

        print("\n❌ SCALING ERROR")

        print(e)

        return None


    # ========================================================
    # CONDITION PREDICTION
    # ========================================================

    try:

        condition_encoded = (

            condition_model.predict(
                scaled_data
            )

        )

    except Exception as e:

        print(
            "\n❌ CONDITION PREDICTION ERROR"
        )

        print(e)

        return None


    # ========================================================
    # DECODE CONDITION
    # ========================================================

    try:

        condition = (

            label_encoder.inverse_transform(
                condition_encoded
            )[0]

        )

    except Exception as e:

        print(
            "\n❌ LABEL DECODING ERROR"
        )

        print(e)

        return None


    # ========================================================
    # RUL PREDICTION
    # ========================================================

    try:

        rul_prediction = (

            life_model.predict(
                scaled_data
            )

        )


        rul = float(
            rul_prediction[0]
        )


        # Prevent negative RUL

        rul = max(
            0,
            rul
        )


    except Exception as e:

        print(
            "\n❌ RUL PREDICTION ERROR"
        )

        print(e)

        return None


    # ========================================================
    # RESULT
    # ========================================================

    print("\nProduct Information")

    print("-" * 50)


    print(
        f"Product              : "
        f"{product_name}"
    )


    print(
        f"Manufacturing Date   : "
        f"{manufacturing_date}"
    )


    print(
        f"Installation Date    : "
        f"{installation_date}"
    )


    print(
        f"Manufacturing Age    : "
        f"{manufacturing_age_years:.2f} years"
    )


    print(
        f"Installation Age     : "
        f"{installation_age_years:.2f} years"
    )


    print("\nPrediction")

    print("-" * 50)


    print(
        f"Quality / Condition  : "
        f"{condition}"
    )


    print(
        f"RUL                  : "
        f"{rul:.2f} months"
    )


    # ========================================================
    # RETURN RESULT
    # ========================================================

    return {

        "product":
            product_name,

        "manufacturing_date":
            manufacturing_date,

        "installation_date":
            installation_date,

        "manufacturing_age_years":
            manufacturing_age_years,

        "installation_age_years":
            installation_age_years,

        "voltage":
            voltage,

        "current":
            load_current,

        "temperature":
            temperature,

        "vibration":
            vibration,

        "condition":
            condition,

        "rul":
            rul

    }


# ============================================================
# PRODUCT 1 PREDICTION
# ============================================================

result1 = predict_product(

    "PRODUCT 1",

    product1_current

)


# ============================================================
# PRODUCT 2 PREDICTION
# ============================================================

result2 = predict_product(

    "PRODUCT 2",

    product2_current

)


# ============================================================
# PRODUCT 3 PREDICTION
# ============================================================

result3 = predict_product(

    "PRODUCT 3",

    product3_current

)


# ============================================================
# DISPLAY FINAL PRODUCT RESULT
# ============================================================

def display_product_result(
    result
):


    if result is None:

        return


    print("\n")

    print("=" * 70)

    print(
        f"📦 {result['product']}"
    )

    print("=" * 70)


    # --------------------------------------------------------
    # PRODUCT DATES
    # --------------------------------------------------------

    print(
        f"Manufacturing Date : "
        f"{result['manufacturing_date']}"
    )


    print(
        f"Installation Date  : "
        f"{result['installation_date']}"
    )


    print(
        f"Manufacturing Age  : "
        f"{result['manufacturing_age_years']:.2f} years"
    )


    print(
        f"Installation Age   : "
        f"{result['installation_age_years']:.2f} years"
    )


    print("-" * 70)


    # --------------------------------------------------------
    # SENSOR VALUES
    # --------------------------------------------------------

    print(
        f"Voltage            : "
        f"{result['voltage']:.2f} V"
    )


    print(
        f"Load Current       : "
        f"{result['current']:.2f}"
    )


    print(
        f"Temperature        : "
        f"{result['temperature']:.2f} °C"
    )


    print(
        f"Vibration          : "
        f"{result['vibration']:.4f}"
    )


    print("-" * 70)


    # --------------------------------------------------------
    # ML RESULTS
    # --------------------------------------------------------

    print(
        f"Quality / Condition: "
        f"{result['condition']}"
    )


    print(
        f"RUL                : "
        f"{result['rul']:.2f} months"
    )


# ============================================================
# FINAL RESULTS
# ============================================================

print("\n\n")

print("#" * 70)

print(
    "📊 FINAL PRODUCT QUALITY & RUL RESULTS"
)

print("#" * 70)


# ============================================================
# PRODUCT 1
# ============================================================

display_product_result(
    result1
)


# ============================================================
# PRODUCT 2
# ============================================================

display_product_result(
    result2
)


# ============================================================
# PRODUCT 3
# ============================================================

display_product_result(
    result3
)


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n")

print("=" * 70)

print("📋 FINAL SUMMARY")

print("=" * 70)


def print_summary(result):

    if result is None:

        return


    print(

        f"{result['product']} → "

        f"Condition: "
        f"{result['condition']} | "

        f"RUL: "
        f"{result['rul']:.2f} months | "

        f"Manufactured: "
        f"{result['manufacturing_date']} | "

        f"Installed: "
        f"{result['installation_date']}"

    )


print_summary(result1)

print_summary(result2)

print_summary(result3)


# ============================================================
# COMPLETION MESSAGE
# ============================================================

print("\n")

print("=" * 70)

print(
    "✅ ALL THREE PRODUCTS PROCESSED SUCCESSFULLY"
)

print("=" * 70)

print("\n")


LOADING COMMON ML MODELS
✅ Scaler loaded successfully
✅ Label Encoder loaded successfully
✅ Condition Model loaded successfully
✅ RUL / Life Model loaded successfully

FETCHING COMMON SENSOR VALUES
✅ Common Voltage (V0)       : 193.0
✅ Common Temperature (V4)   : 31.0
✅ Common Vibration (V5)     : 0.0

FETCHING PRODUCT CURRENT VALUES
✅ Product 1 Current (V1)    : 173.0
✅ Product 2 Current (V2)    : 276.0
✅ Product 3 Current (V3)    : 442.0

COMMON SENSOR VALUES
Voltage       : 193.00 V
Temperature   : 31.00 °C
Vibration     : 0.0000

PRODUCT LOAD CURRENT VALUES
Product 1 Current : 173.00
Product 2 Current : 276.00
Product 3 Current : 442.00


----------------------------------------------------------------------
🔮 PREDICTING PRODUCT 1
----------------------------------------------------------------------

Model Input
--------------------------------------------------
Voltage       : 193.00
Load Current  : 173.00
Temperature   : 31.00
Vibration     : 0.0000

Product Information
-------